# Stage 04 - Publish Test Findings

Publish replay scope, data-feed health, decision posture, recovery accountability, and attached finding records for leadership and analyst views.

In [ ]:
from pyspark.sql import Window, functions as F

rule_hits_df = spark.table('silver_stream_rule_hits')
measures_df = spark.table('silver_streaming_measures')
expectations_df = spark.table('silver_stream_expectations')
bronze_events_df = spark.table('bronze_recorded_observed_events')
preservation_df = spark.table('bronze_replay_preservation_metadata')

gold_table_names = {
    'summary': 'gold_stream_health_summary',
    'current': 'gold_current_test_findings',
    'history': 'gold_test_findings_history',
    'evidence': 'gold_stream_finding_evidence',
    'execution': 'gold_execution_decision_summary',
}

In [ ]:
preservation_context_df = preservation_df.select(
    'replay_manifest_id',
    'replay_digest',
    'observed_row_count',
    'preserved_event_count',
    'event_type_counts_json',
)

current_findings_df = (
    rule_hits_df
    .dropDuplicates(['finding_id', 'finding_version', 'evidence_key'])
    .crossJoin(preservation_context_df)
    .withColumn('finding_run_key', F.concat_ws('|', 'finding_id', 'finding_version', 'evidence_key'))
    .withColumn('first_detected_at_utc', F.col('finding_event_time_utc'))
    .withColumn('published_at_utc', F.current_timestamp())
    .withColumn('decision_class', F.when(F.col('severity') == 'high', F.lit('DIRECTOR REVIEW')).otherwise(F.lit('WATCH')))
    .withColumn('recovery_owner',
        F.when(F.col('finding_code') == 'source_dropout', F.lit('Sensor Data Lead'))
        .when(F.col('finding_code') == 'late_event', F.lit('Battle Management Data Lead'))
        .when(F.col('finding_code') == 'schema_violation', F.lit('Stream Contract Lead'))
        .when(F.col('finding_code') == 'duplicate_event', F.lit('Stream Ingestion Lead'))
        .otherwise(F.lit('Test Control'))
    )
    .withColumn('required_action',
        F.when(F.col('finding_code') == 'source_dropout', F.lit('Assess TPY2 continuity and objective impact'))
        .when(F.col('finding_code') == 'late_event', F.lit('Confirm command path latency is acceptable'))
        .when(F.col('finding_code') == 'schema_violation', F.lit('Quarantine invalid record and confirm source contract'))
        .when(F.col('finding_code') == 'duplicate_event', F.lit('Confirm canonical record retained once'))
        .otherwise(F.lit('Adjudicate before the next decision brief'))
    )
    .withColumn('action_due', F.lit('Before next director decision brief'))
    .select(
        'finding_id',
        'finding_version',
        'finding_status',
        'ruleset_id',
        'ruleset_version',
        'rule_id',
        'rule_version',
        'finding_code',
        'finding_category',
        'severity',
        'decision_class',
        'finding_title',
        'finding_summary',
        'recovery_owner',
        'required_action',
        'action_due',
        'source_table',
        'measure_name',
        'measured_value_text',
        'threshold_text',
        'evidence_key',
        'evidence_event_ids',
        'affected_ordering_key',
        'finding_run_key',
        'first_detected_at_utc',
        'published_at_utc',
        'replay_manifest_id',
        'replay_digest',
        'observed_row_count',
        'preserved_event_count',
        'event_type_counts_json',
        'evidence_json',
    )
)

evidence_df = (
    current_findings_df.select(
        'finding_id',
        'finding_code',
        'rule_id',
        'finding_version',
        'source_table',
        'evidence_key',
        F.explode_outer('evidence_event_ids').alias('evidence_event_id'),
    )
    .join(
        bronze_events_df.select(
            F.col('event_id').alias('joined_event_id'),
            'event_type',
            'source_system',
            'source_instance_id',
            'ordering_key',
            'event_time_utc',
            'ingest_time_utc',
            'bronze_record_sha256',
        ),
        F.col('evidence_event_id') == F.col('joined_event_id'),
        'left',
    )
    .withColumn('evidence_status', F.when(F.col('joined_event_id').isNull(), F.lit('MISSING_REFERENCED_EVENT')).otherwise(F.lit('ATTACHED')))
    .select(
        'finding_id',
        'finding_code',
        'rule_id',
        'finding_version',
        'source_table',
        'evidence_key',
        'evidence_event_id',
        F.col('joined_event_id').alias('event_id'),
        'event_type',
        'source_system',
        'source_instance_id',
        'ordering_key',
        'event_time_utc',
        'ingest_time_utc',
        'bronze_record_sha256',
        'evidence_status',
    )
)


In [ ]:
current_history_df = current_findings_df.withColumn('history_first_published_at_utc', F.col('published_at_utc'))

if spark.catalog.tableExists(gold_table_names['history']):
    existing_history_df = spark.table(gold_table_names['history'])
else:
    existing_history_df = current_history_df.limit(0)

combined_history_df = existing_history_df.unionByName(current_history_df, allowMissingColumns=True)
history_window = Window.partitionBy('finding_run_key').orderBy(F.col('history_first_published_at_utc').asc_nulls_last(), F.col('published_at_utc').asc_nulls_last())
history_df = (
    combined_history_df
    .withColumn('history_rank', F.row_number().over(history_window))
    .filter(F.col('history_rank') == 1)
    .drop('history_rank')
)

summary_df = (
    measures_df.groupBy('source_system')
    .agg(
        F.max('event_time_ts').alias('latest_event_time_utc'),
        F.max('ingest_lag_seconds').alias('max_ingest_lag_seconds'),
        F.max('sequence_gap_count').alias('max_sequence_gap_count'),
        F.max('continuity_gap_seconds').alias('max_continuity_gap_seconds'),
        F.sum(F.when(F.col('is_late_event'), 1).otherwise(0)).alias('late_event_count'),
        F.sum(F.when((F.col('event_type') == 'sensor.observation') & (F.col('has_sequence_gap') | F.col('has_continuity_gap') | F.col('low_continuity_signal')), 1).otherwise(0)).alias('dropout_signal_count'),
        F.sum(F.when(F.col('event_type') == 'preservation.marker', 1).otherwise(0)).alias('preservation_marker_count'),
    )
    .crossJoin(preservation_context_df.select('replay_manifest_id', 'replay_digest'))
)

execution_scope_df = bronze_events_df.agg(
    F.first('test_event_id', ignorenulls=True).alias('test_event_id'),
    F.concat_ws(', ', F.sort_array(F.collect_set(F.when(F.col('site_id') != '', F.col('site_id'))))).alias('sites'),
    F.count(F.lit(1)).alias('raw_record_count'),
    F.countDistinct('event_type').alias('event_type_count'),
    F.countDistinct('source_system').alias('source_system_count'),
    F.max(F.to_timestamp('event_time_utc')).alias('observation_window_end_utc'),
    F.max(F.to_timestamp('ingest_time_utc')).alias('latest_ingest_time_utc'),
)
pending_ack_df = expectations_df.filter(F.col('expectation_type') == 'FOLLOW_ON_EVENT').agg(
    F.first('expectation_status', ignorenulls=True).alias('acknowledgment_status'),
    F.first('due_by_utc', ignorenulls=True).alias('acknowledgment_due_utc'),
    F.first('target_instance_id', ignorenulls=True).alias('acknowledgment_target'),
)
finding_posture_df = current_findings_df.agg(
    F.count(F.lit(1)).alias('open_finding_count'),
    F.sum(F.when(F.col('decision_class') == 'DIRECTOR REVIEW', 1).otherwise(0)).alias('director_review_count'),
    F.sum(F.when(F.col('decision_class') == 'WATCH', 1).otherwise(0)).alias('watch_count'),
)
execution_df = (
    execution_scope_df
    .crossJoin(preservation_context_df.select('preserved_event_count'))
    .crossJoin(pending_ack_df)
    .crossJoin(finding_posture_df)
    .withColumn('execution_mode', F.lit('DETERMINISTIC REPLAY'))
    .withColumn('classification_label', F.lit('SYNTHETIC UNCLASS'))
    .withColumn('test_phase', F.lit('RUNNING THE TEST - REPLAY COMPLETE'))
    .withColumn('decision_recommendation', F.lit('NO DECISION - DIRECTOR REVIEW REQUIRED'))
    .withColumn('decision_authority', F.lit('Not assigned in synthetic demo'))
    .withColumn('scope_note', F.lit('Full replay scope; no report selections applied'))
)

current_findings_df.cache()
evidence_df.cache()
history_df.cache()
summary_df.cache()
execution_df.cache()
current_findings_df.count()
evidence_df.count()
history_df.count()
summary_df.count()
execution_df.count()

for table_name, frame in {
    gold_table_names['current']: current_findings_df,
    gold_table_names['history']: history_df,
    gold_table_names['evidence']: evidence_df,
    gold_table_names['summary']: summary_df,
    gold_table_names['execution']: execution_df,
}.items():
    (
        frame.write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(table_name)
    )

In [ ]:
print(f"Current findings: {spark.table(gold_table_names['current']).count()}")
print(f"History rows after idempotent merge: {spark.table(gold_table_names['history']).count()}")
spark.table(gold_table_names['current']).orderBy('finding_code', 'finding_id').show(truncate=False)
spark.table(gold_table_names['history']).select('finding_code', 'finding_id', 'finding_version', 'finding_run_key', 'history_first_published_at_utc').orderBy('finding_code', 'finding_id').show(truncate=False)
spark.table(gold_table_names['summary']).orderBy('source_system').show(truncate=False)
spark.table(gold_table_names['execution']).show(truncate=False)
spark.table(gold_table_names['evidence']).orderBy('finding_id', 'evidence_event_id', 'bronze_record_sha256').show(truncate=False)
